# CTA Pipeline — Dry Run

Quick end-to-end smoke test for **pretraining** and **fine-tuning**.

All cells run from the **UTUEL repo root** — make sure the kernel working directory is set there.

```
Stage 1 — Pretrain  : TableEmbedJePA on CTA data (SMP regime)
Stage 2 — Finetune  : multi-label column type classifier
```

In [1]:
import os, sys
from pathlib import Path

# ── Ensure we run from the repo root ─────────────────────────────────────────
REPO_ROOT = Path("__file__").resolve().parent.parent  # CTA/../
if not (REPO_ROOT / "CTA").exists():
    # Fallback: assume notebook is already opened from repo root
    REPO_ROOT = Path.cwd()
os.chdir(REPO_ROOT)
print(f"Working directory: {Path.cwd()}")

# Add repo root + TRL-model to path so imports resolve
for p in [str(REPO_ROOT), str(REPO_ROOT / "TRL-model")]:
    if p not in sys.path:
        sys.path.insert(0, p)

Working directory: C:\Users\wtchuitc\Documents\GitHub\UTUEL


## 1 — Shared Hydra overrides for a fast dry run

These CLI flags cap records/rows/epochs so the whole notebook completes in seconds.

In [1]:
# Override flags appended to every %run command
DRY = (
    "data.max_records=20 "
    "data.max_rows_train=5 "
    "data.max_rows_dev=1 "
    "data.max_rows_test=1 "
	"data.folder=D:\\TABLE_DATASET\\HYTREL\\ckpt_data\\ckpt_data\\data\\col_ann "
    "pretraining.epochs=10 "
    "pretraining.batch_size=4 "
    "finetuning.epochs=10 "
    "finetuning.batch_size=4 "
    "pretraining.dataloader_num_workers=0 "
    "finetuning.dataloader_num_workers=0 "
    "embedder.cache_embeddings=true "
)
print("Dry-run overrides:", DRY)


Dry-run overrides: data.max_records=20 data.max_rows_train=5 data.max_rows_dev=1 data.max_rows_test=1 data.folder=D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann pretraining.epochs=10 pretraining.batch_size=4 finetuning.epochs=10 finetuning.batch_size=4 pretraining.dataloader_num_workers=0 finetuning.dataloader_num_workers=0 embedder.cache_embeddings=true 


In [ ]:
CUDA_VISIBLE_DEVICES=0 python3 CTA/pretain.py data.max_rows_train=10 data.max_rows_dev=1 data.max_rows_test=1  pretraining.epochs=10 pretraining.batch_size=4 finetuning.epochs=10 finetuning.batch_size=4 pretraining.dataloader_num_workers=0 finetuning.dataloader_num_workers=0 embedder.cache_embeddings=true embedder.model_type=ollama embedder.model_name=all-minilm

## 1b — Dataset inspection

Peek at raw records and what `CTASMPDataset` produces before running any training.


In [6]:
import json, sys, os
from pathlib import Path
from omegaconf import OmegaConf

# ── Resolve the data folder the same way finetune.py does ────────────────────
sys.path.insert(0, str(Path.cwd() / "CTA"))
from dataset_utils import resolve_data_paths

# Parse the DRY overrides to extract data.folder / paths
dry_cfg_overrides = [kv for kv in DRY.split() if kv.startswith("data.")]
base_cfg = OmegaConf.load("config.yaml")
override_cfg = OmegaConf.from_dotlist(dry_cfg_overrides)
data_cfg = OmegaConf.merge(base_cfg, override_cfg).data

_paths = resolve_data_paths(data_cfg)
print("Resolved paths:")
for k, v in _paths.items():
    print(f"  {k:12s} → {v}  (exists={Path(v).exists()})")

# ── Load and peek at raw records ─────────────────────────────────────────────
MAX_RECORDS = 5
MAX_ROWS    = 5   # same cap as the dry-run overrides
train_path  = _paths["train"]
raw = json.loads(Path(train_path).read_text(encoding="utf-8"))[:MAX_RECORDS]

print(f"\nFirst {MAX_RECORDS} records from train set ({Path(train_path).name}):")
for i, rec in enumerate(raw):
    table_id  = rec[0]
    caption   = rec[1]
    headers   = rec[5] if len(rec) > 5 else []
    col_types = rec[7] if len(rec) > 7 else []
    cell_links = rec[6] if len(rec) > 6 else []
    n_cells = sum(len(cp) for cp in cell_links)
    print(f"\n  [{i}] id={table_id!r}  caption={caption!r}")
    print(f"       headers ({len(headers)}): {headers}")
    print(f"       col_types ({len(col_types)} cols): {[t[:2] for t in col_types]}")
    print(f"       n_cells (entity links): {n_cells}")

# ── Inspect CTASMPDataset output ─────────────────────────────────────────────
sys.path.insert(0, str(Path.cwd() / "TRL-model"))
from dataset import CTASMPDataset
from dataset import _reconstruct_table   # helper used internally

print(f"\n── Building CTASMPDataset (train, max_records={MAX_RECORDS}, max_rows={MAX_ROWS}) ──")
ds = CTASMPDataset(
    data_path=train_path,
    model_type=base_cfg.embedder.model_type,
    model_name=base_cfg.embedder.model_name,
    max_records=MAX_RECORDS,
    max_rows_per_table=MAX_ROWS,
    precompute=False,   # skip embedding — just check structure
)

print(f"\nRecords loaded : {len(ds.records)}")
print(f"U-paths total  : {len(ds._samples)}")

# ── Build a col_type lookup from raw records (table_id → list[list[str]]) ────
col_types_lookup: dict[str, list[list[str]]] = {
    str(r[0]): (r[7] if len(r) > 7 else []) for r in raw
}

# ── Per-table: show cropped table + first 3 U-paths with labels ──────────────
SHOW_N = 3   # U-paths to show per table

for ri, rec_dict in enumerate(ds.records[:MAX_RECORDS]):
    table_id = rec_dict["table_id"]
    header   = rec_dict["header"]
    rows     = rec_dict["rows"]
    col_types_for_table = col_types_lookup.get(table_id, [])

    print(f"\n{'═'*70}")
    print(f"  Record {ri}  table_id={table_id!r}")
    print(f"{'═'*70}")

    # ── Cropped table ─────────────────────────────────────────────────────
    col_w = [max(len(str(h)), 12) for h in header]
    for r_row in rows:
        for ci, cell in enumerate(r_row):
            col_w[ci] = max(col_w[ci], min(len(str(cell)), 30))

    def _fmt(vals):
        return "  " + "  │  ".join(str(v)[:30].ljust(col_w[ci]) for ci, v in enumerate(vals))

    header_label = [f"{h} [{col_types_for_table[ci][0] if ci < len(col_types_for_table) and col_types_for_table[ci] else '—'}]"
                    for ci, h in enumerate(header)]
    print("  CROPPED TABLE (header [label] + data rows):")
    print(_fmt(header_label))
    print("  " + "─" * (sum(col_w) + 5 * len(col_w)))
    for r_row in rows:
        print(_fmt(r_row))

    # ── U-path samples for this record ────────────────────────────────────
    rec_samples = [(j, up) for j, (ridx, up) in enumerate(ds._samples) if ridx == ri]
    print(f"\n  U-paths for this table: {len(rec_samples)}")
    print(f"  Showing first {min(SHOW_N, len(rec_samples))}:\n")
    for j, up in rec_samples[:SHOW_N]:
        label_a = (col_types_for_table[up.col_idx_a][0]
                   if up.col_idx_a < len(col_types_for_table) and col_types_for_table[up.col_idx_a]
                   else "—")
        label_b = (col_types_for_table[up.col_idx_b][0]
                   if up.col_idx_b < len(col_types_for_table) and col_types_for_table[up.col_idx_b]
                   else "—")
        print(f"    [sample {j}]")
        print(f"      node_id  pivot_a={up.pivot_a}  node_a={up.node_a}  node_b={up.node_b}  pivot_b={up.pivot_b}")
        print(f"      col_a    idx={up.col_idx_a}  header={up.col_header_a!r}  label={label_a!r}")
        print(f"      col_b    idx={up.col_idx_b}  header={up.col_header_b!r}  label={label_b!r}")
        print(f"      cell_a   {up.cell_value_a!r}")
        print(f"      cell_b   {up.cell_value_b!r}")
        print(f"      SMP text {up.smp_text!r}")
        print(f"      BAR text {up.reversed_smp_text!r}")
        print()


Resolved paths:
  train        → D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann\train.table_col_type.json  (exists=True)
  dev          → D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann\dev.table_col_type.json  (exists=True)
  test         → D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann\test.table_col_type.json  (exists=True)
  type_vocab   → D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann\type_vocab.txt  (exists=True)

First 5 records from train set (train.table_col_type.json):

  [0] id='2728176-1'  caption='slovak parliamentary election, 1990'
       headers (1): ['party']
       col_types (1 cols): [['organization.organization']]
       n_cells (entity links): 9

  [1] id='2728185-1'  caption='czechoslovakian parliamentary election, 1946'
       headers (1): ['party']
       col_types (1 cols): [['government.political_party', 'organization.organization']]
       n_cells (entity links): 8

  [2] id='27282030-1'  caption='1961 tanfl season'
    

ImportError: cannot import name 'CTASMPDataset' from 'dataset' (c:\Users\wtchuitc\Documents\GitHub\UTUEL\TRL-model\dataset.py)

## 1c — Inspect `extract_column_embeddings` step-by-step

Runs the embedding extraction logic (no training) and prints per-column details:
which U-paths contributed, which side (a/b), the raw label vector, and the resulting embedding shape.


In [ ]:
import sys, json
from collections import defaultdict
from pathlib import Path
from omegaconf import OmegaConf

# ── Imports (reuse already-resolved paths from cell above) ───────────────────
sys.path.insert(0, str(Path.cwd() / "CTA"))
sys.path.insert(0, str(Path.cwd() / "TRL-model"))

import torch
import torch.nn.functional as F
from dataset_utils import load_type_vocab, resolve_data_paths
from dataset import CTASMPDataset   # CTA/dataset.py

base_cfg = OmegaConf.load("config.yaml")
dry_cfg_overrides = [kv for kv in DRY.split() if kv.startswith("data.")]
override_cfg = OmegaConf.from_dotlist(dry_cfg_overrides)
data_cfg = OmegaConf.merge(base_cfg, override_cfg).data
paths = resolve_data_paths(data_cfg)

# ── Build CTASMPDataset (with embeddings) ────────────────────────────────────
MAX_RECORDS = 5
MAX_ROWS    = 6   # +1 for header

print("Building CTASMPDataset (precompute=True) …")
smp_ds = CTASMPDataset(
    data_path=paths["train"],
    model_type=base_cfg.embedder.model_type,
    model_name=base_cfg.embedder.get("model_name"),
    max_records=MAX_RECORDS,
    max_rows_per_table=MAX_ROWS,
    precompute=True,
    cache_embeddings=base_cfg.embedder.cache_embeddings,
    embed_cache_dir=base_cfg.embedder.get("embed_cache_dir"),
)
print(f"  {len(smp_ds.records)} tables  {len(smp_ds._samples)} U-paths  embed_cache shape={smp_ds._embed_cache.shape}")

# ── Load type vocab & col_types lookup ───────────────────────────────────────
type2idx, idx2type = load_type_vocab(paths["type_vocab"])
raw_records = json.loads(Path(paths["train"]).read_text(encoding="utf-8"))[:MAX_RECORDS]
col_types_per_table = {str(r[0]): (r[7] if len(r) > 7 else []) for r in raw_records}

# ── Replicate extract_column_embeddings index-building ───────────────────────
rec_col_a_to_js = defaultdict(list)
rec_col_b_to_js = defaultdict(list)
for j, (rec_idx, up) in enumerate(smp_ds._samples):
    rec_col_a_to_js[(rec_idx, up.col_idx_a)].append(j)
    if up.col_idx_b != up.col_idx_a:
        rec_col_b_to_js[(rec_idx, up.col_idx_b)].append(j)

# ── Walk through each table/column and print details ─────────────────────────
SHOW_COLS = 3   # columns to inspect per table

for rec_idx, rec_dict in enumerate(smp_ds.records):
    table_id = rec_dict["table_id"]
    header   = rec_dict["header"]
    col_types = col_types_per_table.get(table_id, [])

    print(f"\n{'═'*72}")
    print(f"  Record {rec_idx}  table_id={table_id!r}  columns={len(header)}")
    print(f"{'═'*72}")

    shown = 0
    for col_idx, col_hdr in enumerate(header):
        if shown >= SHOW_COLS:
            break

        types_for_col = col_types[col_idx] if col_idx < len(col_types) else []
        if not types_for_col:
            print(f"  col {col_idx} ({col_hdr!r})  → no type annotation — skip")
            continue

        # Build multi-hot
        hot = torch.zeros(len(type2idx))
        for t in types_for_col:
            if t in type2idx:
                hot[type2idx[t]] = 1.0
        active_labels = [t for t in types_for_col if t in type2idx]
        if hot.sum() == 0:
            print(f"  col {col_idx} ({col_hdr!r})  → types {types_for_col} not in vocab — skip")
            continue

        js_a = rec_col_a_to_js.get((rec_idx, col_idx), [])
        js_b = rec_col_b_to_js.get((rec_idx, col_idx), [])
        if not js_a and not js_b:
            print(f"  col {col_idx} ({col_hdr!r})  → no U-paths found — skip")
            continue

        print(f"\n  ── col {col_idx}  header={col_hdr!r}  labels={active_labels}")
        print(f"     hot vector (non-zero): { {idx2type[i]: hot[i].item() for i in hot.nonzero(as_tuple=True)[0].tolist()} }")
        print(f"     U-paths on a-side: {len(js_a)}  │  b-side: {len(js_b)}")

        # Show contributing U-paths
        for side, js, node_slot in [("a-side (node_a=smp[:,1])", js_a, 1),
                                     ("b-side (node_b=smp[:,2])", js_b, 2)]:
            if not js:
                continue
            print(f"\n     {side}:")
            for j in js[:3]:
                _, up = smp_ds._samples[j]
                txt = up.cell_value_a if node_slot == 1 else up.cell_value_b
                partner_hdr = up.col_header_b if node_slot == 1 else up.col_header_a
                partner_cell = up.cell_value_b if node_slot == 1 else up.cell_value_a
                partner_label = (col_types[up.col_idx_b][0] if node_slot == 1 and up.col_idx_b < len(col_types) and col_types[up.col_idx_b]
                                 else col_types[up.col_idx_a][0] if node_slot == 2 and up.col_idx_a < len(col_types) and col_types[up.col_idx_a]
                                 else "—")
                idx_t = torch.tensor([j], dtype=torch.long)
                raw_emb = smp_ds._embed_cache[smp_ds._smp_idx[idx_t, node_slot]]  # [1, d]
                print(f"       sample {j}: cell={txt!r:25s}  partner_hdr={partner_hdr!r}({partner_label})  partner_cell={partner_cell!r}")
                print(f"                  embed norm={raw_emb.norm().item():.4f}  shape={tuple(raw_emb.shape)}")
            if len(js) > 3:
                print(f"       … {len(js)-3} more")

        # Simulate final pooled embedding
        reps = []
        if js_a:
            idx_a = torch.tensor(js_a, dtype=torch.long)
            reps.append(F.normalize(smp_ds._embed_cache[smp_ds._smp_idx[idx_a, 1]], dim=-1))
        if js_b:
            idx_b = torch.tensor(js_b, dtype=torch.long)
            reps.append(F.normalize(smp_ds._embed_cache[smp_ds._smp_idx[idx_b, 2]], dim=-1))
        cell_reps = torch.cat(reps, dim=0)
        col_emb   = cell_reps.mean(dim=0)
        print(f"\n     → pooled column embedding: shape={tuple(col_emb.shape)}  "
              f"norm={col_emb.norm().item():.4f}  "
              f"from {len(cell_reps)} node reps")

        shown += 1


## 2 — Stage 1: Pretrain

In [ ]:
PRETRAIN_OUT = "checkpoints/dry_run_pretrain"
%run pretrain.py {DRY} pretraining.output_dir={PRETRAIN_OUT}

Seed set to 42


matmul_precision: high
model:
  hidden_size: 384
  num_layers: 3
  num_heads: 12
  intermediate_size: null
  attention_dropout: 0.1
  hidden_dropout: 0.1
  layer_norm_eps: 1.0e-12
  temperature: 0.07
  beta: 0.5
  ema_decay: 0.996
  ablate_proj: true
loss:
  jepa: 1.0
  jepa_bar: 1.0
  local: 1.0
  global: 1.0
smp:
  use_graph_walks: false
  num_walks: 50
  chunk_size: 1
classifier:
  pool_mode: mean
  freeze_encoder: false
  embed_mode: column
  intermediate_size: null
  include_header_emb: true
  smp_source: smp
data:
  folder: D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann
  train_path: datasets/train.table_col_type.json
  dev_path: datasets/dev.table_col_type.json
  test_path: datasets/test.table_col_type.json
  type_vocab_path: datasets/type_vocab.txt
  max_records: 20
  max_rows_train: 5
  max_rows_dev: 1
  max_rows_test: 1
embedder:
  model_type: huggingface
  base_url: http://127.0.0.1:11434/
  model_name: sentence-transformers/all-MiniLM-L6-v2
  api_key: null
  preco

Using 16bit Automatic Mixed Precision (AMP)


[CTA][pretrain] dataset=datasets/train.table_col_type.json  tables=20  u-paths=116  embed_dim=384  hidden_size=384
[CTA][pretrain] embedder slug  : all-MiniLM-L6-v2
[CTA][pretrain] checkpoint dir : checkpoints\dry_run_pretrain\all-MiniLM-L6-v2\2026-06-17_20-11-47_5cdf8942


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Trainable params: 6.5 M                                                                                            
Non-trainable params: 5.3 M                                                                                        
Total params: 11.8 M                                                                                               
Total estimated model params size (MB): 47.327                                                                     
Modules in train mode: 119                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


`Trainer.fit` stopped: `max_epochs=10` reached.


[CTA][pretrain] finished  train_loss=0.5813


In [2]:
PRETRAIN_OUT = "checkpoints/dry_run_pretrain/all-MiniLM-L6-v2/2026-06-17_20-11-47_5cdf8942"
from pathlib import Path
# Verify a checkpoint was written
ckpts = sorted(Path(PRETRAIN_OUT).glob("*.ckpt"))
print(f"Checkpoints found: {[c.name for c in ckpts]}")
PRETRAIN_CKPT = str(ckpts[-1]) if ckpts else None
print(f"Using checkpoint : {PRETRAIN_CKPT}")

Checkpoints found: ['cta-pretrain-epoch=07-train_loss=0.7213.ckpt', 'cta-pretrain-epoch=08-train_loss=0.6519.ckpt', 'cta-pretrain-epoch=09-train_loss=0.5813.ckpt', 'last.ckpt']
Using checkpoint : checkpoints\dry_run_pretrain\all-MiniLM-L6-v2\2026-06-17_20-11-47_5cdf8942\last.ckpt


## 3 — Stage 2a: Finetune without pretrained encoder (raw LLM embeddings)

In [ ]:
FT_OUT_RAW = "CTA/checkpoints/dry_run_finetune_raw"
%run finetune.py {DRY} \
    finetuning.pretrained_ckpt=null \
    finetuning.output_dir={FT_OUT_RAW} \
    eval.output_dir=CTA/outputs/dry_run_raw \
    classifier.smp_source=smp \
    classifier.embed_mode=cell

## 3 — Stage 2b: Finetune with pretrained encoder from Stage 1

In [5]:
# Override flags appended to every %run command
DRY = (
    " data.max_records=null "
    " data.max_rows_train=5 "
    " data.max_rows_dev=1 "
    " data.max_rows_test=1 "
	" data.folder=D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann "
    " pretraining.epochs=50 "
    " pretraining.batch_size=64 "
    " finetuning.epochs=20 "
    " finetuning.batch_size=1024 "
    "finetuning.lr=1e-2 "
    "finetuning.weight_decay=0.2 "
    " pretraining.dataloader_num_workers=0 "
    " finetuning.dataloader_num_workers=0 "
    " embedder.cache_embeddings=true "
	" embedder.embed_cache_dir=D:/UTUEL_OUTPUT/cache_embeddings/Cache_CTA "
    " embedder.model_type=ollama "
     " embedder.model_name=all-minilm "
    # " embedder.model_name=nomic-embed-text "
	# "classifier.intermediate_size=255 "
    "model.hidden_size=384 "
)
print("Dry-run overrides:", DRY)

FT_OUT_ENC = "checkpoints/dry_run_finetune_enc "
PRETRAIN_CKPT = "D:/UTUEL_OUTPUT/all-minilm/2026-06-24_16-34-32_9c3ee648 "
# PRETRAIN_CKPT = "D:/UTUEL_OUTPUT/nomic-embed-text/2026-06-24_16-43-30_c9f91484 "
if PRETRAIN_CKPT:
    %run finetune.py {DRY} \
        finetuning.pretrained_ckpt={PRETRAIN_CKPT} \
        finetuning.output_dir={FT_OUT_ENC} \
        eval.output_dir=CTA/outputs/dry_run_enc \
        classifier.smp_source=smp \
        classifier.embed_mode=cell_header \
		classifier.freeze_encoder=false \
		eval.threshold=0.5
else:
    print("No pretrain checkpoint found — skipping Stage 2b")

Dry-run overrides:  data.max_records=null  data.max_rows_train=5  data.max_rows_dev=1  data.max_rows_test=1  data.folder=D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann  pretraining.epochs=50  pretraining.batch_size=64  finetuning.epochs=20  finetuning.batch_size=1024 finetuning.lr=1e-2 finetuning.weight_decay=0.2  pretraining.dataloader_num_workers=0  finetuning.dataloader_num_workers=0  embedder.cache_embeddings=true  embedder.embed_cache_dir=D:/UTUEL_OUTPUT/cache_embeddings/Cache_CTA  embedder.model_type=ollama  embedder.model_name=all-minilm model.hidden_size=384 


Seed set to 42


matmul_precision: high
model:
  hidden_size: 384
  num_layers: 3
  num_heads: 12
  intermediate_size: null
  attention_dropout: 0.1
  hidden_dropout: 0.1
  layer_norm_eps: 1.0e-12
  temperature: 0.07
  beta: 0.5
  ema_decay: 0.996
  ablate_proj: true
loss:
  jepa: 0.0
  jepa_bar: 0.0
  local: 1.0
  global: 0.0
smp:
  use_graph_walks: false
  num_walks: 50
  chunk_size: 1
classifier:
  pool_mode: mean
  freeze_encoder: false
  embed_mode: cell_header
  intermediate_size: null
  pos_weight: null
  pos_weight_max: 100.0
  include_header_emb: true
  smp_source: smp
data:
  folder: D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann
  train_path: datasets/train.table_col_type.json
  dev_path: datasets/dev.table_col_type.json
  test_path: datasets/test.table_col_type.json
  type_vocab_path: datasets/type_vocab.txt
  max_records: null
  max_rows_train: 5
  max_rows_dev: 1
  max_rows_test: 1
embedder:
  model_type: ollama
  base_url: http://127.0.0.1:11434/
  model_name: all-minilm
  api_

c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\lightning_fabric\utilities\cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


[CTA][finetune] loaded pretrained encoder from D:\UTUEL_OUTPUT\all-minilm\2026-06-24_16-34-32_9c3ee648\last.ckpt
  [OllamaEmbedder] could not auto-detect context length: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>; falling back to 4,096 chars
[CTA][SMP] 397098 tables → 3523716 U-paths from train.table_col_type.json
[CTA][SMP][cache] file=D:\UTUEL_OUTPUT\cache_embeddings\Cache_CTA\train.table_col_type_all-minilm_smp.embed_cache.pt enabled=True samples=3523716


...

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test/accuracy       │    0.9979945421218872     │
│       test/f1_macro       │    0.2979913055896759     │
│       test/f1_micro       │    0.8636396527290344     │
│         test/loss         │   0.005749459378421307    │
│   test/precision_macro    │    0.3155565857887268     │
│   test/precision_micro    │    0.8779364824295044     │
│     test/recall_macro     │    0.29991358518600464    │
│     test/recall_micro     │     0.849995493888855     │
└───────────────────────────┴───────────────────────────┘

[CTA][finetune][dev]  accuracy=0.9981  precision_micro=0.8799  recall_micro=0.8576  f1_micro=0.8686  precision_macro=0.6141  recall_macro=0.5368  f1_macro=0.5462  (thr=0.5)  -> CTA\outputs\dry_run_enc\all-minilm\2026-06-29_22-45-26_03968df5\cta_dev_metrics.json
[CTA][finetune][test]  accuracy=0.9980  precision_micro=0.8780  recall_micro=0.8497  f1_micro=0.8636  precision_macro=0.6385  recall_macro=0.5263  f1_macro=0.5503  (thr=0.5)  -> CTA\outputs\dry_run_enc\all-minilm\2026-06-29_22-45-26_03968df5\cta_test_metrics.json


## 4 — Ablation: smp_source and embed_mode variants

Quick sweep over the two key classifier config axes using raw embeddings (fastest).

In [5]:
import json
from itertools import product

results = {}

for smp_src, emb_mode in product(["smp", "smp_bar", "both"], ["column", "cell"]):
    tag = f"{smp_src}_{emb_mode}"
    out_dir = f"CTA/outputs/dry_run_{tag}"
    ckpt_dir = f"CTA/checkpoints/dry_run_{tag}"
    print(f"\n{'='*60}\n  smp_source={smp_src}  embed_mode={emb_mode}\n{'='*60}")
    %run CTA/finetune.py {DRY} \
        finetuning.pretrained_ckpt=null \
        finetuning.output_dir={ckpt_dir} \
        eval.output_dir={out_dir} \
        classifier.smp_source={smp_src} \
        classifier.embed_mode={emb_mode}
    # collect dev metrics
    m_path = Path(out_dir) / "cta_dev_metrics.json"
    if m_path.exists():
        results[tag] = json.loads(m_path.read_text())

print("\nAblation summary (dev):")
for tag, m in results.items():
    print(f"  {tag:25s}  acc={m.get('accuracy',0):.4f}  f1_micro={m.get('f1_micro',0):.4f}  f1_macro={m.get('f1_macro',0):.4f}")


  smp_source=smp  embed_mode=column


Exception: File `'CTA/finetune.py'` not found.

In [ ]:
%reload_ext autoreload
%autoreload 2
from pathlib import Path
import re

# ── Resolve checkpoint paths ──────────────────────────────────────────────────
# _PT  = str(sorted(Path("checkpoints/dry_run_pretrain").glob("*.ckpt"))[-1]) \
#        if list(Path("checkpoints/dry_run_pretrain").glob("*.ckpt")) else ""
# _FTE = str(sorted(Path("checkpoints/dry_run_finetune_enc").glob("*.ckpt"))[-1]) \
#        if list(Path("checkpoints/dry_run_finetune_enc").glob("*.ckpt")) else ""

_FTE = "C:/Users/wtchuitc/Documents/GitHub/UTUEL/CTA/checkpoints/dry_run_finetune_enc/all-minilm/2026-06-29_12-56-20_cc864854/last.ckpt"
_PT="D:/UTUEL_OUTPUT/all-minilm/2026-06-24_16-34-32_9c3ee648/last.ckpt"

# ── Extract data folder from DRY (avoids OmegaConf backslash issues) ─────────
_m = re.search(r"data\.folder=(\S+)", DRY)
_DATA_FOLDER = _m.group(1) if _m else ""

# ── Pull embedder + data params straight from DRY so cache & model match ─────
def _dry(key, default=""):
    m = re.search(rf"{re.escape(key)}=(\S+)", DRY)
    return m.group(1) if m else default

_MODEL_TYPE  = _dry("embedder.model_type",     "ollama")
_MODEL_NAME  = _dry("embedder.model_name",     "all-minilm")
_CACHE_DIR   = _dry("embedder.embed_cache_dir", "")
_MAX_RECORDS = _dry("data.max_records",        "20")
_MAX_ROWS    = _dry("data.max_rows_test",      "6")
print(f"Resolved data folder: {_DATA_FOLDER}")
print(f"Embedder: type={_MODEL_TYPE} name={_MODEL_NAME} cache={_CACHE_DIR}")
%run visualize_embeddings.py \
    --pretrain_ckpt    {_PT} \
    --ft_enc_ckpt      {_FTE} \
    --data_folder      {_DATA_FOLDER} \
    --model_type       {_MODEL_TYPE} \
    --model_name       {_MODEL_NAME} \
    --embed_cache_dir  {_CACHE_DIR} \
    --split            test \
    --max_records      {_MAX_RECORDS} \
    --max_rows         {_MAX_ROWS} \
    --top_k_labels     12


Resolved data folder: D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann
Embedder: type=ollama name=all-minilm cache=D:/UTUEL_OUTPUT/cache_embeddings/Cache_CTA
[CTA][vocab] loaded 255 types from type_vocab.txt
[viz] building CTASMPDataset  split=test  max_records=10000  max_rows=1
  [OllamaEmbedder] could not auto-detect context length: <urlopen error [WinError 10061] No connection could be made because the target machine actively refused it>; falling back to 4,096 chars
[CTA][SMP] 4764 tables → 14650 U-paths from test.table_col_type.json
[CTA][SMP][cache] file=D:\UTUEL_OUTPUT\cache_embeddings\Cache_CTA\test.table_col_type_all-minilm_smp.embed_cache.pt enabled=True samples=14650
[CTA][SMP][cache] loaded test.table_col_type_all-minilm_smp.embed_cache.pt doc_texts=11047 qry_texts=23876
[viz] 4764 tables  14650 U-paths  embed_cache shape=torch.Size([11047, 384])
[viz] extracting  Raw (no encoder) …
       node pts=57546  col pts=13025
[viz] extracting  After pretrain …
       node p

c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(
c:\Users\wtchuitc\anaconda3\envs\in_context_learning\lib\site-packages\umap\spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is 

KeyboardInterrupt: 

## 5 — Inspect dev / test metric JSON

In [7]:
import json

for split in ("dev", "test"):
    p = Path("CTA/outputs/dry_run_raw") / f"cta_{split}_metrics.json"
    if p.exists():
        print(f"\n── {split} metrics (raw embeddings) ──")
        print(json.dumps(json.loads(p.read_text()), indent=2))
    else:
        print(f"{p} not found")


── dev metrics (raw embeddings) ──
{
  "accuracy": 0.9936704635620117,
  "precision_micro": 0.6666666865348816,
  "recall_micro": 0.19607843458652496,
  "f1_micro": 0.3030303120613098,
  "precision_macro": 0.006593618076294661,
  "recall_macro": 0.005698108114302158,
  "f1_macro": 0.0060648005455732346,
  "threshold": 0.5
}

── test metrics (raw embeddings) ──
{
  "accuracy": 0.9953725337982178,
  "precision_micro": 0.8846153616905212,
  "recall_micro": 0.29113924503326416,
  "f1_micro": 0.43809524178504944,
  "precision_macro": 0.015406163409352303,
  "recall_macro": 0.011251168325543404,
  "f1_macro": 0.012324931100010872,
  "threshold": 0.5
}


## 6 — Inspect labels (`target_in`) for train / dev / test

Loads the multi-hot **target** matrix each split feeds to the BCE loss
(`target_in` = `multi_hot` of shape `[N, num_classes]`) and reports, per split:
label cardinality, how many classes are present, the most frequent labels,
and a few example rows decoded back to type names.


In [1]:
import sys, json
from pathlib import Path
from omegaconf import OmegaConf
import torch
import pandas as pd
from collections import defaultdict, Counter

sys.path.insert(0, str(Path.cwd() / "CTA"))
sys.path.insert(0, str(Path.cwd() / "TRL-model"))

from dataset_utils import load_type_vocab, resolve_data_paths
from dataset import CTASMPDataset
from finetune import extract_column_embeddings

TOP_K = 15

# Override flags appended to every %run command
DRY = (
    " data.max_records=20000 "
    " data.max_rows_train=10 "
    " data.max_rows_dev=1 "
    " data.max_rows_test=1 "
	" data.folder=D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann "
    " pretraining.epochs=20 "
    " pretraining.batch_size=64 "
    " finetuning.epochs=50 "
    " finetuning.batch_size=64 "
    " pretraining.dataloader_num_workers=0 "
    " finetuning.dataloader_num_workers=0 "
    " embedder.cache_embeddings=true "
	" embedder.embed_cache_dir=D:/UTUEL_OUTPUT/cache_embeddings/Cache_CTA "
    " embedder.model_type=ollama "
    " embedder.model_name=all-minilm "
	"classifier.embed_mode=cell_header "
)
print("Dry-run overrides:", DRY)

# ── Hyperparameters: reuse the DRY overrides defined above ───────────────────
print("Using DRY overrides:", DRY)

# ── Resolve config from the DRY overrides (data + embedder + classifier) ──────
base_cfg = OmegaConf.load("config.yaml")
_ov = [kv for kv in DRY.split()
       if kv.startswith(("data.", "embedder.", "classifier."))]
cfg = OmegaConf.merge(base_cfg, OmegaConf.from_dotlist(_ov))
paths = resolve_data_paths(cfg.data)
type2idx, idx2type = load_type_vocab(paths["type_vocab"])
num_classes = len(type2idx)

# ── Use the classifier config from DRY (falls back to config.yaml defaults) ──
EMBED_MODE = cfg.classifier.embed_mode   # 'column' | 'cell'
SMP_SOURCE = cfg.classifier.smp_source   # 'smp' | 'smp_bar' | 'both'
print(f"type_vocab: {num_classes} classes  |  embed_mode={EMBED_MODE}  smp_source={SMP_SOURCE}\n")

def _col_types(p):
    recs = json.loads(Path(p).read_text(encoding="utf-8"))
    mr = cfg.data.get("max_records")
    return {str(r[0]): (r[7] if len(r) > 7 else []) for r in (recs[:mr] if mr else recs)}

_MAX_ROWS = {"train": cfg.data.get("max_rows_train"),
             "dev":   cfg.data.get("max_rows_dev"),
             "test":  cfg.data.get("max_rows_test")}

# ── Build the target (multi-hot) matrix for each split ───────────────────────
targets       = {}   # split -> multi_hot tensor [N, num_classes]
label_sets    = {}   # split -> set of active class indices
struct_stats  = {}   # split -> dict with header-only / single-column counts
oov_counter   = Counter()   # out-of-vocab type name -> # columns it caused to skip
for split in ("train", "dev", "test"):
    smp_ds = CTASMPDataset(
        data_path=paths[split],
        model_type=cfg.embedder.model_type,
        base_url=cfg.embedder.get("base_url"),
        model_name=cfg.embedder.get("model_name"),
        api_key=cfg.embedder.get("api_key"),
        max_records=cfg.data.get("max_records"),
        max_rows_per_table=_MAX_ROWS[split],
        precompute=True,
        cache_embeddings=cfg.embedder.cache_embeddings,
        embed_cache_dir=cfg.embedder.get("embed_cache_dir"),
    )

    # ── Structural checks on the loaded tables ───────────────────────────────
    #   header_only  : table has no data rows (only the column header)
    #   single_col   : table has exactly one column
    n_tables    = len(smp_ds.records)
    header_only = sum(1 for r in smp_ds.records if len(r.get("rows", [])) == 0)
    single_col  = sum(1 for r in smp_ds.records if len(r.get("header", [])) == 1)
    struct_stats[split] = {
        "n_tables":    n_tables,
        "header_only": header_only,
        "single_col":  single_col,
    }

    # ── Per-reason skip breakdown (mirrors extract_column_embeddings) ─────────
    # Rebuild the a-side / b-side U-path indices, apply the smp_source filter,
    # then categorise every annotated column exactly as the extractor does.
    _col_types_split = _col_types(paths[split])
    rec_col_a = defaultdict(list)
    rec_col_b = defaultdict(list)
    for j, (ri, up) in enumerate(smp_ds._samples):
        rec_col_a[(ri, up.col_idx_a)].append(j)
        if up.col_idx_b != up.col_idx_a:
            rec_col_b[(ri, up.col_idx_b)].append(j)

    kept = skip_oov = skip_nopath = 0
    nopath_header_only = nopath_single_col = nopath_other = 0
    nopath_last_col = nopath_first_col = 0   # a/b-side asymmetry from smp_source
    for rec_idx, rec in enumerate(smp_ds.records):
        table_id   = str(rec.get("table_id", ""))
        n_cols     = len(rec.get("header", []))
        n_rows     = len(rec.get("rows", []))
        types_tbl  = _col_types_split.get(table_id, [])
        for col_idx, types_for_col in enumerate(types_tbl):
            if not types_for_col:
                continue                          # no annotation → not a "skip"
            in_vocab = [t for t in types_for_col if t in type2idx]
            if not in_vocab:                      # reason 1: types not in vocab
                skip_oov += 1
                for t in types_for_col:
                    oov_counter[t] += 1
                continue
            js_a = rec_col_a.get((rec_idx, col_idx), [])
            js_b = rec_col_b.get((rec_idx, col_idx), [])
            if SMP_SOURCE == "smp":
                js_b = []
            elif SMP_SOURCE == "smp_bar":
                js_a = []
            if not js_a and not js_b:             # reason 2: no usable U-paths
                skip_nopath += 1
                if n_rows == 0:
                    nopath_header_only += 1
                elif n_cols == 1:
                    nopath_single_col += 1
                else:
                    nopath_other += 1
                # a/b-side asymmetry: with smp_source='smp' the LAST column is
                # never an a-side; with 'smp_bar' the FIRST column is never b-side
                if n_cols > 1 and col_idx == n_cols - 1:
                    nopath_last_col += 1
                if n_cols > 1 and col_idx == 0:
                    nopath_first_col += 1
                continue
            kept += 1
    struct_stats[split].update({
        "kept": kept, "skip_oov": skip_oov, "skip_nopath": skip_nopath,
        "nopath_header_only": nopath_header_only,
        "nopath_single_col":  nopath_single_col,
        "nopath_other":       nopath_other,
        "nopath_last_col":    nopath_last_col,
        "nopath_first_col":   nopath_first_col,
    })
    print(f"       skip breakdown → kept={kept}  out_of_vocab={skip_oov}  "
          f"no_upaths={skip_nopath} (header_only={nopath_header_only} "
          f"single_col={nopath_single_col} other={nopath_other})")

    _embs, multi_hot, col_ids = extract_column_embeddings(
        smp_ds, type2idx, _col_types(paths[split]),
        embed_mode=EMBED_MODE, smp_source=SMP_SOURCE,
    )
    targets[split]    = multi_hot
    freq              = multi_hot.sum(dim=0)                       # positives per class
    label_sets[split] = set(freq.nonzero(as_tuple=True)[0].tolist())
    per_sample        = multi_hot.sum(dim=1)
    struct_stats[split].update({
        "n_samples":       int(multi_hot.shape[0]),
        "classes_present": len(label_sets[split]),
        "labels_per_smp":  float(per_sample.mean()),
    })

# ── Out-of-vocab types that caused columns to be skipped ─────────────────────
print(f"\n{'='*64}\nOUT-OF-VOCAB TYPES (skipped columns, across all splits)\n{'='*64}")
print(f"  distinct out-of-vocab types: {len(oov_counter)}")
# ╭──────────────────────────────────────────────────────────────────────────╮
# │ Summary tables — rendered as DataFrames for easier inspection            │
# ╰──────────────────────────────────────────────────────────────────────────╯
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

def _names(idxs):
    return sorted(idx2type[i] for i in idxs)

# ── 1) Per-split structure + skip breakdown ──────────────────────────────────
summary_rows = []
for split in ("train", "dev", "test"):
    s = struct_stats[split]
    summary_rows.append({
        "split":           split,
        "tables":          s["n_tables"],
        "header_only":     s["header_only"],
        "single_col":      s["single_col"],
        "kept_cols":       s["kept"],
        "skip_oov":        s["skip_oov"],
        "skip_no_upaths":  s["skip_nopath"],
        "nopath_hdr_only": s["nopath_header_only"],
        "nopath_1col":     s["nopath_single_col"],
        "nopath_other":    s["nopath_other"],
        "nopath_last":     s["nopath_last_col"],
        "nopath_first":    s["nopath_first_col"],
        "samples":         s.get("n_samples", 0),
        "classes":         f"{s.get('classes_present', 0)}/{num_classes}",
        "labels/smp":      round(s.get("labels_per_smp", 0.0), 2),
    })
summary_df = pd.DataFrame(summary_rows).set_index("split")
print(f"SKIP & STRUCTURE SUMMARY  [embed_mode={EMBED_MODE}  smp_source={SMP_SOURCE}]")
display(summary_df)

oov_df = (pd.DataFrame(oov_counter.most_common(TOP_K),
                       columns=["oov_type", "skipped_cols"])
          .set_index("oov_type"))
print(f"\nOUT-OF-VOCAB TYPES (top {TOP_K} of {len(oov_counter)})")
display(oov_df)

# ── 3) Target label overlap across splits (pairwise Jaccard) ─────────────────
tr, dv, te = label_sets["train"], label_sets["dev"], label_sets["test"]
common_all = tr & dv & te
_sets = {"train": tr, "dev": dv, "test": te}
overlap_rows = []
for a in ("train", "dev", "test"):
    row = {"split": a, "n_classes": len(_sets[a])}
    for b in ("train", "dev", "test"):
        inter, union = _sets[a] & _sets[b], _sets[a] | _sets[b]
        row[f"J({b})"] = round(len(inter) / len(union), 3) if union else 0.0
    overlap_rows.append(row)
overlap_df = pd.DataFrame(overlap_rows).set_index("split")
print(f"\nTARGET LABEL OVERLAP — Jaccard between splits "
      f"(train∩dev∩test = {len(common_all)} classes)")
display(overlap_df)

# ── 4) Classes in dev/test never seen in train (unlearnable) ─────────────────
unseen_rows = []
for split, st in (("dev", dv), ("test", te)):
    unseen = st - tr
    unseen_rows.append({
        "split":        split,
        "unseen_count": len(unseen),
        "examples":     ", ".join(_names(unseen)[:TOP_K]) + (" …" if len(unseen) > TOP_K else ""),
    })
unseen_df = pd.DataFrame(unseen_rows).set_index("split")

print("\nLABELS NOT IN TRAIN (unlearnable for dev/test)")
display(overlap_df)      
print(f"{', '.join(_names(common_all)[:TOP_K])}")

display(unseen_df)
print(f"\nShared by ALL three splits ({len(common_all)}): {', '.join(_names(common_all)[:TOP_K])}")

Dry-run overrides:  data.max_records=20000  data.max_rows_train=10  data.max_rows_dev=1  data.max_rows_test=1  data.folder=D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann  pretraining.epochs=20  pretraining.batch_size=64  finetuning.epochs=50  finetuning.batch_size=64  pretraining.dataloader_num_workers=0  finetuning.dataloader_num_workers=0  embedder.cache_embeddings=true  embedder.embed_cache_dir=D:/UTUEL_OUTPUT/cache_embeddings/Cache_CTA  embedder.model_type=ollama  embedder.model_name=all-minilm classifier.embed_mode=cell_header 
Using DRY overrides:  data.max_records=20000  data.max_rows_train=10  data.max_rows_dev=1  data.max_rows_test=1  data.folder=D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann  pretraining.epochs=20  pretraining.batch_size=64  finetuning.epochs=50  finetuning.batch_size=64  pretraining.dataloader_num_workers=0  finetuning.dataloader_num_workers=0  embedder.cache_embeddings=true  embedder.embed_cache_dir=D:/UTUEL_OUTPUT/cache_embeddings/Cache

c:\Users\wtchuitc\Documents\GitHub\UTUEL\CTA\dataset.py:630: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(cache_file, map_location="cpu")


[CTA][SMP][cache] stale train.table_col_type_all-minilm_smp.embed_cache.pt (smp_idx rows=4496889 vs samples=293299; embed rows=1206575) — recomputing
[CTA][SMP][embed] 106384 unique documents + 0 unique queries …
[CTA][SMP][embed] 106384/106384 documents not in cache — embedding …


[CTA][SMP][embed] documents:   0%|          | 0/1663 [00:00<?, ?batch/s]

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

## 7 — Merge per-split caches into one global document cache

Collects **every unique document text** (cell + header strings) across the
`train` / `dev` / `test` SMP caches and stores them in a **single entry keyed by
model name** — `global_{model_name}_smp.embed_cache.pt`.

The merge reuses the embeddings already computed in the existing
`*_smp.embed_cache.pt` files (no re-embedding); each text is deduplicated so a
cell/header shared across splits is kept once.


In [ ]:
import re
import torch
import pandas as pd
from pathlib import Path

# ── Resolve cache dir + model name straight from the DRY overrides ───────────
def _dry(key, default=""):
    m = re.search(rf"{re.escape(key)}=(\S+)", DRY)
    return m.group(1) if m else default

CACHE_DIR  = Path(_dry("embedder.embed_cache_dir", "."))
MODEL_NAME = _dry("embedder.model_name", _dry("embedder.model_type", "all-minilm"))
MODEL_SLUG = MODEL_NAME.replace("/", "-")
GLOBAL_CACHE = CACHE_DIR / f"global_{MODEL_SLUG}_smp.embed_cache.pt"

# ── Find every per-split SMP cache for this model (skip the global file) ─────
candidates = sorted(
    p for p in CACHE_DIR.glob(f"*{MODEL_SLUG}*_smp.embed_cache.pt")
    if p.name != GLOBAL_CACHE.name
)
print(f"Cache dir : {CACHE_DIR}")
print(f"Model     : {MODEL_NAME}")
print(f"Found {len(candidates)} per-split cache file(s):")
for p in candidates:
    print(f"  • {p.name}")

# ── Merge: dedup document texts, keep the first embedding seen per text ──────
text_to_idx: dict[str, int] = {}
vecs: list[torch.Tensor] = []
embed_dim = None
per_file = []
for p in candidates:
    ckpt = torch.load(p, map_location="cpu")
    doc_t2i = ckpt.get("doc_text_to_idx", ckpt.get("text_to_idx"))
    doc_emb = ckpt.get("doc_embed_cache", ckpt.get("embed_cache"))
    if doc_t2i is None or doc_emb is None:
        print(f"  ! {p.name}: no document cache — skipped")
        continue
    if embed_dim is None:
        embed_dim = int(doc_emb.shape[1])
    elif int(doc_emb.shape[1]) != embed_dim:
        print(f"  ! {p.name}: dim {doc_emb.shape[1]} != {embed_dim} — skipped")
        continue
    added = 0
    for txt, idx in doc_t2i.items():
        if txt not in text_to_idx:
            text_to_idx[txt] = len(vecs)
            vecs.append(doc_emb[idx])
            added += 1
    per_file.append((p.name, len(doc_t2i), added))

if not vecs:
    raise RuntimeError(
        "No document embeddings found to merge — run the per-split cells first "
        "so the *_smp.embed_cache.pt files exist."
    )

doc_embed_cache = torch.stack(vecs)  # [n_unique, d]

# ── Store everything in a SINGLE entry keyed by model name ───────────────────
CACHE_DIR.mkdir(parents=True, exist_ok=True)
torch.save({
    "doc_embed_cache": doc_embed_cache,
    "doc_text_to_idx": text_to_idx,
    "embed_dim":       int(doc_embed_cache.shape[1]),
    "model_name":      MODEL_NAME,
    "merged_from":     [p.name for p in candidates],
}, GLOBAL_CACHE)

# ── Report ───────────────────────────────────────────────────────────────────
merge_df = (pd.DataFrame(per_file, columns=["cache_file", "doc_texts", "new_unique"])
            .set_index("cache_file"))
display(merge_df)
print(f"\nMerged {len(per_file)} cache(s) → {len(text_to_idx)} unique document texts "
      f"(dim={doc_embed_cache.shape[1]})")
print(f"Saved single global cache → {GLOBAL_CACHE}  "
      f"({GLOBAL_CACHE.stat().st_size / 1024**2:.1f} MB)")


Cache dir : D:\UTUEL_OUTPUT\cache_embeddings\Cache_CTA
Model     : nomic-embed-text
Found 3 per-split cache file(s):
  • dev.table_col_type_nomic-embed-text_smp.embed_cache.pt
  • test.table_col_type_nomic-embed-text_smp.embed_cache.pt
  • train.table_col_type_nomic-embed-text_smp.embed_cache.pt


C:\Users\wtchuitc\AppData\Local\Temp\ipykernel_22080\3708934855.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(p, map_location="cpu")


  ! dev.table_col_type_nomic-embed-text_smp.embed_cache.pt: dim 384 != 768 — skipped
  ! train.table_col_type_nomic-embed-text_smp.embed_cache.pt: dim 384 != 768 — skipped


,doc_texts,new_doc,new_qry
cache_file,,,
test.table_col_type_nomic-embed-text_smp.embed_cache.pt,11047,11047,23876



Merged 1 cache(s) → 11047 unique document texts + 23876 unique query texts (dim=768)
Saved single global cache → D:\UTUEL_OUTPUT\cache_embeddings\Cache_CTA\global_nomic-embed-text_smp.embed_cache.pt  (104.0 MB)


## 8 — Build SMPs and pull their cell embeddings from the global cache

Loads the dataset, generates its **SMP / U-paths**, then for every U-path looks
up the 4 token texts `[pivot_a, node_a, node_b, pivot_b]` in the
`global_{model_name}_smp.embed_cache.pt` file and gathers their embeddings —
**no re-embedding**, everything comes from the merged cache. Reports how many
tokens were found vs. missing (coverage).


In [ ]:
import sys, re
import torch
from pathlib import Path
from omegaconf import OmegaConf

sys.path.insert(0, str(Path.cwd() / "CTA"))
sys.path.insert(0, str(Path.cwd() / "TRL-model"))
from dataset_utils import resolve_data_paths
from dataset import CTASMPDataset

# ── 1) Load the global document cache  (text → idx → embedding) ──────────────
def _dry(key, default=""):
    m = re.search(rf"{re.escape(key)}=(\S+)", DRY)
    return m.group(1) if m else default

CACHE_DIR    = Path(_dry("embedder.embed_cache_dir", "."))
MODEL_NAME   = _dry("embedder.model_name", _dry("embedder.model_type", "all-minilm"))
MODEL_SLUG   = MODEL_NAME.replace("/", "-")
GLOBAL_CACHE = CACHE_DIR / f"global_{MODEL_SLUG}_smp.embed_cache.pt"

gck         = torch.load(GLOBAL_CACHE, map_location="cpu")
text_to_idx = gck["doc_text_to_idx"]          # {str: int}
embeds      = gck["doc_embed_cache"]          # [n_unique, d]
d_emb       = int(embeds.shape[1])
print(f"Global cache: {GLOBAL_CACHE.name}  texts={len(text_to_idx)}  dim={d_emb}")

# ── 2) Load dataset + generate SMP / U-paths (no embedding) ──────────────────
base_cfg = OmegaConf.load("config.yaml")
_ov  = [kv for kv in DRY.split() if kv.startswith(("data.", "embedder."))]
cfg  = OmegaConf.merge(base_cfg, OmegaConf.from_dotlist(_ov))
paths = resolve_data_paths(cfg.data)

SPLIT = "train"
ds = CTASMPDataset(
    data_path=paths[SPLIT],
    model_type=cfg.embedder.model_type,
    model_name="nomic-embed-text", #cfg.embedder.get("model_name"),
    max_records=cfg.data.get("max_records"),
    max_rows_per_table=cfg.data.get(f"max_rows_{SPLIT}"),
    precompute=False,        # build structure + U-paths only; embeddings come from cache
)
print(f"{SPLIT}: {len(ds.records)} tables  {len(ds._samples)} U-paths")

# ── 3) Map every U-path's 4 token texts → row index in the global cache ──────
MISS = -1
idx_rows, miss_examples = [], []
found = miss = 0
for _, up in ds._samples:
    tokens = (up.col_header_a, up.cell_value_a, up.cell_value_b, up.col_header_b)
    row = []
    for t in tokens:
        i = text_to_idx.get(t, MISS)
        row.append(i)
        if i == MISS:
            miss += 1
            if len(miss_examples) < 10:
                miss_examples.append(t)
        else:
            found += 1
    idx_rows.append(row)

smp_idx = torch.tensor(idx_rows, dtype=torch.long)        # [N, 4]

# ── 4) Gather embeddings from the cache (zero-fill any misses) ───────────────
N    = smp_idx.shape[0]
flat = smp_idx.reshape(-1)                                # [N*4]
ok   = flat != MISS
gathered = torch.zeros(flat.shape[0], d_emb)
gathered[ok] = embeds[flat[ok]]
smp_embs = gathered.reshape(N, 4, d_emb)                  # [N, 4, d]  → [pivot_a, node_a, node_b, pivot_b]

total = found + miss
print(f"\nToken lookups: {found}/{total} found "
      f"({miss} missing, {100 * found / max(total, 1):.1f}% coverage)")
print(f"SMP embedding tensor: {tuple(smp_embs.shape)}  (= [N, 4, d])")
if miss_examples:
    print("Missing examples:", miss_examples)

# ── Peek: first U-path tokens + their cached-embedding norms ─────────────────
if N:
    _, up0 = ds._samples[0]
    print("\nFirst U-path tokens & cached embed norms:")
    for slot, name, txt in zip(
        range(4),
        ["pivot_a", "node_a", "node_b", "pivot_b"],
        [up0.col_header_a, up0.cell_value_a, up0.cell_value_b, up0.col_header_b],
    ):
        print(f"  [{slot}] {name:8s} {txt!r:30s} norm={smp_embs[0, slot].norm().item():.4f}")


C:\Users\wtchuitc\AppData\Local\Temp\ipykernel_22080\2366676459.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gck         = torch.load(GLOBAL_CACHE, map_location="cpu

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\UTUEL_OUTPUT\\cache_embeddings\\Cache_CTA\\global_nomic-embed-text_smp.embed_cache.pt'

In [ ]:
# ── Target-output (multi-label) combination overlap: train vs dev vs test ────
# For every (table, column) with a type annotation we treat the FULL set of
# vocab types on that column as ONE unique multi-label combination, then count
# how often each combination appears in the train / dev / test split and report
# how the dev & test combinations overlap with the train split.
import json
from collections import Counter
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf
from dataset_utils import load_type_vocab, resolve_data_paths

# Use the FULL train/dev/test files (ignore dry-run max_records/max_rows truncation).
_base_cfg = OmegaConf.load("config.yaml")

# set the path folder to D:\TABLE_DATASET\HYTREL\ckpt_data\ckpt_data\data\col_ann
_base_cfg.data.folder = "D:/TABLE_DATASET/HYTREL/ckpt_data/ckpt_data/data/col_ann"
_paths = resolve_data_paths(_base_cfg.data)
type2idx, _ = load_type_vocab(_paths["type_vocab"])


def _combo_counter(json_path: str | Path) -> Counter:
    """Count unique multi-label combinations over all annotated columns.

    Combination key = sorted tuple of the column's types that are in the vocab.
    Columns with no in-vocab type are skipped (they carry no training label).
    """
    records = json.loads(Path(json_path).read_text(encoding="utf-8"))
    counter: Counter = Counter()
    for rec in records:
        col_types = rec[7] if len(rec) > 7 else []
        for types_for_col in col_types:
            combo = tuple(sorted(t for t in types_for_col if t in type2idx))
            if combo:
                counter[combo] += 1
    return counter


train_counts = _combo_counter(_paths["train"])
dev_counts   = _combo_counter(_paths["dev"])
test_counts  = _combo_counter(_paths["test"])

all_combos = sorted(
    set(train_counts) | set(dev_counts) | set(test_counts),
    key=lambda c: (
        -(train_counts.get(c, 0) + dev_counts.get(c, 0) + test_counts.get(c, 0)),
        c,
    ),
)

combo_df = pd.DataFrame(
    {
        "combination": [" | ".join(c) for c in all_combos],
        "n_labels":    [len(c) for c in all_combos],
        "train_count": [train_counts.get(c, 0) for c in all_combos],
        "dev_count":   [dev_counts.get(c, 0) for c in all_combos],
        "test_count":  [test_counts.get(c, 0) for c in all_combos],
    }
)
combo_df["in_train"] = combo_df["train_count"] > 0
combo_df["in_dev"]   = combo_df["dev_count"]   > 0
combo_df["in_test"]  = combo_df["test_count"]  > 0
# Does this dev/test combination also exist in the train split?
combo_df["in_train_dev_test"] = combo_df["in_train"] & combo_df["in_dev"] & combo_df["in_test"]

train_set, dev_set, test_set = set(train_counts), set(dev_counts), set(test_counts)

# Overlap of dev / test combinations WITH the train split.
dev_in_train    = len(dev_set & train_set)
dev_not_train   = len(dev_set - train_set)
test_in_train   = len(test_set & train_set)
test_not_train  = len(test_set - train_set)
n_multi         = int((combo_df["n_labels"] > 1).sum())

print("Unique multi-label combinations")
print(f"  train      : {len(train_set):4d}  ({sum(train_counts.values())} annotated columns)")
print(f"  dev        : {len(dev_set):4d}  ({sum(dev_counts.values())} annotated columns)")
print(f"  test       : {len(test_set):4d}  ({sum(test_counts.values())} annotated columns)")
print(f"  union      : {len(all_combos):4d}  (of which {n_multi} are true multi-label, n_labels>1)")
print(f"  in all 3   : {len(train_set & dev_set & test_set)}")
print()
print("Overlap with TRAIN (unseen combos are unlearnable):")
print(f"  dev  ∩ train : {dev_in_train:4d} / {len(dev_set)}   (dev  combos NOT in train: {dev_not_train})")
print(f"  test ∩ train : {test_in_train:4d} / {len(test_set)}   (test combos NOT in train: {test_not_train})")

# Combinations that appear in dev/test but were never seen in train.
unseen_df = combo_df[
    (~combo_df["in_train"]) & (combo_df["in_dev"] | combo_df["in_test"])
][["combination", "n_labels", "dev_count", "test_count"]].reset_index(drop=True)
print(f"\nDev/Test combinations missing from TRAIN: {len(unseen_df)}")
display(unseen_df)

display(combo_df)


[CTA][vocab] loaded 255 types from type_vocab.txt
Unique multi-label combinations
  dev        : 423  (13391 annotated columns)
  test       : 450  (13025 annotated columns)
  in both    : 308
  dev only   : 115
  test only  : 142
  total union: 565  (of which 479 are true multi-label, n_labels>1)


,combination,n_labels,dev_count,test_count,in_dev,in_test,in_both
0,location.location,1,1485,1403,True,True,True
1,people.person,1,1168,1137,True,True,True
2,people.person | sports.pro_athlete,2,825,820,True,True,True
3,time.event,1,557,529,True,True,True
4,sports.sports_team,1,551,489,True,True,True
...,...,...,...,...,...,...,...
560,soccer.football_team,1,0,1,False,True,False
561,soccer.football_team | sports.professional_spo...,3,1,0,True,False,False
562,sports.sports_award_type | time.recurring_event,2,0,1,False,True,False
563,sports.sports_league_draft | time.event,2,0,1,False,True,False
